# 01 - Asset Exploration

Primer laboratorio de MacroQuant Lab.

Objetivo: comparar S&P 500, Nasdaq 100, oro, bonos de largo plazo de Estados Unidos y Bitcoin usando retornos, volatilidad, Sharpe, drawdowns y correlaciones.

## 1. Setup

Importamos el codigo del proyecto. La idea importante aqui es separar la logica reutilizable en `src/` y usar el notebook para investigar, visualizar y contar la historia.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from macroquant_lab.config import CORE_ASSET_TICKERS
from macroquant_lab.data_loader import download_prices
from macroquant_lab.metrics import (
    calculate_returns,
    correlation_matrix,
    drawdown_series,
    performance_summary,
    rolling_volatility,
)
from macroquant_lab.visualization import (
    plot_correlation_heatmap,
    plot_cumulative_returns,
    plot_drawdowns,
    plot_prices,
)

CORE_ASSET_TICKERS

## 2. Descargar precios

Usamos precios ajustados, porque incorporan dividendos y splits cuando el proveedor los publica. Para un primer analisis educativo es suficiente; mas adelante revisaremos limitaciones de datos.

In [ ]:
START_DATE = "2015-01-01"

prices = download_prices(CORE_ASSET_TICKERS, start=START_DATE)
prices.tail()

In [ ]:
plot_prices(prices)

## 3. Retornos

El retorno simple diario es:

`r_t = P_t / P_{t-1} - 1`

Esta transformacion es clave: casi todo el analisis cuantitativo trabaja con retornos, no con precios absolutos.

In [ ]:
returns = calculate_returns(prices)
returns.describe().T

In [ ]:
plot_cumulative_returns(returns)

## 4. Tabla de riesgo/rentabilidad

Metricas iniciales:

- `total_return`: retorno total del periodo.
- `annualized_return`: retorno anualizado geometrico.
- `annualized_volatility`: volatilidad diaria escalada por `sqrt(252)`.
- `sharpe_ratio`: retorno por unidad de volatilidad.
- `max_drawdown`: peor caida desde un maximo anterior.

In [ ]:
summary = performance_summary(prices)
summary.style.format({
    "total_return": "{:.1%}",
    "annualized_return": "{:.1%}",
    "annualized_volatility": "{:.1%}",
    "sharpe_ratio": "{:.2f}",
    "max_drawdown": "{:.1%}",
})

## 5. Drawdowns

El drawdown mide cuanto cae un activo desde su maximo previo. Es una metrica muy intuitiva para riesgo real: una cartera puede tener buena rentabilidad media y aun asi ser dificil de aguantar si sus caidas son profundas.

In [ ]:
plot_drawdowns(prices)

In [ ]:
drawdown_series(prices).min().sort_values()

## 6. Correlaciones

La correlacion ayuda a estudiar diversificacion. Si dos activos se mueven de forma muy parecida, combinarlos no reduce mucho el riesgo. Si se comportan distinto, pueden mejorar la estabilidad de una cartera.

In [ ]:
correlation_matrix(returns).round(2)

In [ ]:
plot_correlation_heatmap(returns)

## 7. Volatilidad movil

La volatilidad no es constante. Mirar ventanas moviles permite ver cuando cambia el regimen de mercado.

In [ ]:
rolling_vol = rolling_volatility(returns, window=63)
rolling_vol.tail()

In [ ]:
import plotly.express as px

fig = px.line(rolling_vol, title="Rolling 3-month volatility")
fig.update_layout(yaxis_tickformat=".0%", xaxis_title=None, legend_title_text=None)
fig

## 8. Primeras preguntas para escribir conclusiones

Antes de pasar a macro, intenta responder en texto:

1. Que activo tuvo mejor Sharpe ratio?
2. Que activo tuvo peor drawdown?
3. Oro y bonos diversificaron frente a bolsa?
4. Bitcoin compensa su volatilidad con mayor rentabilidad?
5. Que cambiaria si miras retornos mensuales en lugar de diarios?

Estas respuestas seran la base del primer informe corto del proyecto.